# Restaurant Analytics Agent - Tool Definitions

Convert the restaurant data pipeline into reusable tools for the agent. These tools allow the agent to answer both structured questions about ratings and review counts and qualitative questions about customer experiences.

1. **Business Lookup**: Retrieve rating, review count, rating distribution, and restaurant metadata.
2. **Competitor Comparison**: Compare restaurants by rating, review volume, and rating distribution.
3. **Theme Retrieval**: Use vector search over review text to retrieve reviews related to themes like service, food quality, ambience, price, wait time, or complaints.

In [0]:
# Install the UC AI client (for creating/testing UC functions as tools)
# and numpy (for computing cosine similarity in semantic search)
%pip install unitycatalog-ai[databricks] numpy databricks-vectorsearch
dbutils.library.restartPython()

---
## Configure Source Tables

In [0]:
from pyspark.sql import functions as F
from unitycatalog.ai.core.databricks import DatabricksFunctionClient
import json

# Initialize the Databricks Function Client
uc_client = DatabricksFunctionClient()

# Configuration
CATALOG = "main"
SCHEMA = "aai510_final_agent"

# Source tables
RESTAURANT_METRICS_TABLE = f"{CATALOG}.{SCHEMA}.restaurant_metrics"
RESTAURANTS_TABLE = f"{CATALOG}.{SCHEMA}.restaurants"
REVIEWS_WITH_TEXT_TABLE = f"{CATALOG}.{SCHEMA}.reviews_with_text"
REVIEWS_FOR_VECTOR_TABLE = f"{CATALOG}.{SCHEMA}.reviews_for_vector_search"

print("=" * 60)
print("Configuring agent tool source tables...")
print("=" * 60)

print(f"Restaurant metrics table: {RESTAURANT_METRICS_TABLE}")
print(f"Restaurants table: {RESTAURANTS_TABLE}")
print(f"Review text table: {REVIEWS_WITH_TEXT_TABLE}")
print(f"Vector search source table: {REVIEWS_FOR_VECTOR_TABLE}")

---
## Business Lookup Tool

Objective: Retrieve metrics for a specific restaurant

In [0]:
%sql
-- Drop existing function if it exists
DROP FUNCTION IF EXISTS main.aai510_final_agent.business_lookup;

-- Create SQL UDF that can query tables directly
-- Note: result_limit parameter is advisory only - returns top 10 matches by review count
CREATE OR REPLACE FUNCTION main.aai510_final_agent.business_lookup(
  restaurant_name STRING COMMENT 'Name of the restaurant to search for (case-insensitive partial match)',
  result_limit INT COMMENT 'Advisory maximum (function returns top 10 by review count)'
)
RETURNS STRING
COMMENT 'Retrieve rating, review count, rating distribution, and restaurant metadata for a specific restaurant'
RETURN (
  WITH matching_restaurants AS (
    SELECT
      gmap_id,
      business_name,
      address,
      avg_rating,
      review_count,
      rating_1_count,
      rating_2_count,
      rating_3_count,
      rating_4_count,
      rating_5_count
    FROM main.aai510_final_agent.restaurant_metrics
    WHERE LOWER(business_name) LIKE CONCAT('%', LOWER(restaurant_name), '%')
    ORDER BY review_count DESC
    LIMIT 10
  ),
  result_count AS (
    SELECT COUNT(*) as cnt FROM matching_restaurants
  )
  SELECT
    CASE
      WHEN (SELECT cnt FROM result_count) = 0 THEN
        TO_JSON(
          NAMED_STRUCT(
            'status', 'not_found',
            'message', CONCAT('No restaurant found matching \'', restaurant_name, '\'')
          )
        )
      ELSE
        TO_JSON(
          NAMED_STRUCT(
            'status', 'success',
            'results', COLLECT_LIST(
              NAMED_STRUCT(
                'gmap_id', gmap_id,
                'business_name', business_name,
                'address', address,
                'avg_rating', avg_rating,
                'review_count', review_count,
                'rating_1_count', rating_1_count,
                'rating_2_count', rating_2_count,
                'rating_3_count', rating_3_count,
                'rating_4_count', rating_4_count,
                'rating_5_count', rating_5_count
              )
            )
          )
        )
    END
  FROM matching_restaurants
);


In [0]:
%sql
-- Test the business_lookup SQL UDF
SELECT main.aai510_final_agent.business_lookup('Cesarina', 5) as result

---
## Competitor Comparison Tool

Objective: Compare restaurant performance to another restaurant

In [0]:
%sql
-- Drop existing function if it exists
DROP FUNCTION IF EXISTS main.aai510_final_agent.competitor_comparison;

-- Create SQL UDF for competitor comparison
CREATE OR REPLACE FUNCTION main.aai510_final_agent.competitor_comparison(
  restaurant_1 STRING COMMENT 'Name of the first restaurant to compare',
  restaurant_2 STRING COMMENT 'Name of the second restaurant to compare',
  limit_per_name INT COMMENT 'Advisory maximum (function returns top 5 per restaurant)'
)
RETURNS STRING
COMMENT 'Compare two specific restaurants by rating, review volume, and rating distribution'
RETURN (
  WITH matches_1 AS (
    SELECT
      gmap_id,
      business_name,
      address,
      avg_rating,
      review_count,
      rating_1_count,
      rating_2_count,
      rating_3_count,
      rating_4_count,
      rating_5_count,
      restaurant_1 as search_name
    FROM main.aai510_final_agent.restaurant_metrics
    WHERE LOWER(business_name) LIKE CONCAT('%', LOWER(restaurant_1), '%')
    ORDER BY review_count DESC
    LIMIT 5
  ),
  matches_2 AS (
    SELECT
      gmap_id,
      business_name,
      address,
      avg_rating,
      review_count,
      rating_1_count,
      rating_2_count,
      rating_3_count,
      rating_4_count,
      rating_5_count,
      restaurant_2 as search_name
    FROM main.aai510_final_agent.restaurant_metrics
    WHERE LOWER(business_name) LIKE CONCAT('%', LOWER(restaurant_2), '%')
    ORDER BY review_count DESC
    LIMIT 5
  ),
  all_matches AS (
    SELECT * FROM matches_1
    UNION ALL
    SELECT * FROM matches_2
  ),
  result_count AS (
    SELECT COUNT(*) as cnt FROM all_matches
  )
  SELECT
    CASE
      WHEN (SELECT cnt FROM result_count) = 0 THEN
        TO_JSON(
          NAMED_STRUCT(
            'status', 'not_found',
            'message', CONCAT('No matching restaurants found for \'', restaurant_1, '\' or \'', restaurant_2, '\'')
          )
        )
      ELSE
        TO_JSON(
          NAMED_STRUCT(
            'status', 'success',
            'restaurant_1', restaurant_1,
            'restaurant_2', restaurant_2,
            'results', COLLECT_LIST(
              NAMED_STRUCT(
                'gmap_id', gmap_id,
                'business_name', business_name,
                'address', address,
                'avg_rating', avg_rating,
                'review_count', review_count,
                'rating_1_count', rating_1_count,
                'rating_2_count', rating_2_count,
                'rating_3_count', rating_3_count,
                'rating_4_count', rating_4_count,
                'rating_5_count', rating_5_count,
                'search_name', search_name
              )
            )
          )
        )
    END
  FROM all_matches
);

In [0]:
%sql
-- Test the competitor_comparison SQL UDF
SELECT main.aai510_final_agent.competitor_comparison('Cesarina', 'Parma Cucina Italiana', 3) as result


---
### Prepare Review Text Table for Vector Search

Vector Search needs a stable primary key. This creates a new table with a `review_id`

To keep vector search efficient while making the agent more generalizable, this section indexes all available review text for the two selected comparison restaurants and a random sample of 20 additional San Diego Italian restaurants. This allows the agent to answer the primary comparison question while still supporting theme retrieval for other restaurants in the dataset.

In [0]:
# Create a vector-search-ready review text table with a stable primary key
# Include the two selected restaurants plus a random sample of other restaurants
from pyspark.sql import Window

NUM_RANDOM_RESTAURANTS = 20
MAX_REVIEWS_PER_RANDOM_RESTAURANT = 50

reviews_with_text_df = spark.table(REVIEWS_WITH_TEXT_TABLE)
restaurants_df = spark.table(RESTAURANTS_TABLE)

reviews_joined_df = (
    reviews_with_text_df
    .join(
        restaurants_df.select("gmap_id", "business_name", "address"),
        on="gmap_id",
        how="inner"
    )
    .filter(F.col("text").isNotNull())
    .filter(F.length(F.col("text")) > 10)
)

# Keep all reviews for selected restaurants
selected_reviews_df = (
    reviews_joined_df
    .filter(
        (F.lower(F.col("business_name")).contains("cesarina")) |
        (F.lower(F.col("business_name")).contains("parma"))
    )
)

# Randomly choose additional restaurants, excluding selected restaurants
random_restaurant_ids_df = (
    reviews_joined_df
    .filter(
        ~(
            (F.lower(F.col("business_name")).contains("cesarina")) |
            (F.lower(F.col("business_name")).contains("parma"))
        )
    )
    .select("gmap_id", "business_name")
    .distinct()
    .orderBy(F.rand(seed=42))
    .limit(NUM_RANDOM_RESTAURANTS)
)

# For those random restaurants, take up to N reviews per restaurant
random_reviews_base_df = (
    reviews_joined_df
    .join(
        random_restaurant_ids_df.select("gmap_id"),
        on="gmap_id",
        how="inner"
    )
)

window_by_restaurant = Window.partitionBy("gmap_id").orderBy(F.rand(seed=42))

random_reviews_df = (
    random_reviews_base_df
    .withColumn("random_review_rank", F.row_number().over(window_by_restaurant))
    .filter(F.col("random_review_rank") <= MAX_REVIEWS_PER_RANDOM_RESTAURANT)
    .drop("random_review_rank")
)

# Combine selected restaurants and random sample
reviews_for_vector_df = (
    selected_reviews_df
    .unionByName(random_reviews_df)
    .select(
        F.sha2(
            F.concat_ws(
                "||",
                F.col("gmap_id"),
                F.coalesce(F.col("user_id"), F.lit("unknown_user")),
                F.coalesce(F.col("time").cast("string"), F.lit("unknown_time")),
                F.coalesce(F.col("text"), F.lit(""))
            ),
            256
        ).alias("review_id"),
        "gmap_id",
        "business_name",
        "address",
        "rating",
        "time",
        "text"
    )
    .dropDuplicates(["review_id"])
)

(
    reviews_for_vector_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(REVIEWS_FOR_VECTOR_TABLE)
)

# Change Data Feed is required for Delta Sync Vector Search indexes
spark.sql(f"""
ALTER TABLE {REVIEWS_FOR_VECTOR_TABLE}
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

print(f"Created vector-search source table: {REVIEWS_FOR_VECTOR_TABLE}")
print(f"Total rows: {spark.table(REVIEWS_FOR_VECTOR_TABLE).count():,}")

display(
    spark.table(REVIEWS_FOR_VECTOR_TABLE)
    .groupBy("business_name")
    .agg(F.count("*").alias("review_text_count"))
    .orderBy(F.desc("review_text_count"))
)

---
### Initialize Vector Search Client

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

print("Vector Search client initialized.")

---
### Configure Vector Search Endpoint and Index

In [0]:
# Vector Search configuration
VECTOR_SEARCH_ENDPOINT_NAME = "restaurant_reviews_vs_endpoint"
VECTOR_SEARCH_INDEX_NAME = f"{CATALOG}.{SCHEMA}.restaurant_review_text_index"

# Databricks embedding model endpoint
# If this endpoint is not available in your workspace, replace it with the embedding endpoint your course/workspace provides.
EMBEDDING_MODEL_ENDPOINT_NAME = "databricks-gte-large-en"

print("=" * 60)
print("Configuring Vector Search index...")
print("=" * 60)

print(f"Endpoint: {VECTOR_SEARCH_ENDPOINT_NAME}")
print(f"Index: {VECTOR_SEARCH_INDEX_NAME}")
print(f"Source table: {REVIEWS_FOR_VECTOR_TABLE}")
print(f"Embedding endpoint: {EMBEDDING_MODEL_ENDPOINT_NAME}")

---
### Create or Connect to Vector Search Endpoint

Create an endpoint/index if needed, otherwise connect to the existing one

In [0]:
# Create Vector Search endpoint if needed

try:
    vsc.get_endpoint(VECTOR_SEARCH_ENDPOINT_NAME)
    print(f"✓ Found existing endpoint: {VECTOR_SEARCH_ENDPOINT_NAME}")

except Exception:
    print(f"Creating endpoint: {VECTOR_SEARCH_ENDPOINT_NAME}")
    vsc.create_endpoint(
        name=VECTOR_SEARCH_ENDPOINT_NAME,
        endpoint_type="STANDARD"
    )
    vsc.wait_for_endpoint(VECTOR_SEARCH_ENDPOINT_NAME)
    print(f"✓ Created endpoint: {VECTOR_SEARCH_ENDPOINT_NAME}")

---
### Create or Connect to Vector Search Index

In [0]:
# Delete offline / partial Vector Search index

try:
    print(f"Deleting index: {VECTOR_SEARCH_INDEX_NAME}")
    
    vsc.delete_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=VECTOR_SEARCH_INDEX_NAME
    )
    
    print(f"Deleted index: {VECTOR_SEARCH_INDEX_NAME}")

except Exception as e:
    print("Index delete failed or index was already gone.")
    print(e)

In [0]:
# Create Delta Sync index if needed

try:
    review_index = vsc.get_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=VECTOR_SEARCH_INDEX_NAME
    )
    print(f"✓ Found existing index: {VECTOR_SEARCH_INDEX_NAME}")

except Exception:
    print(f"Creating index: {VECTOR_SEARCH_INDEX_NAME}")
    
    review_index = vsc.create_delta_sync_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=VECTOR_SEARCH_INDEX_NAME,
        source_table_name=REVIEWS_FOR_VECTOR_TABLE,
        pipeline_type="TRIGGERED",
        primary_key="review_id",
        embedding_source_column="text",
        embedding_model_endpoint_name=EMBEDDING_MODEL_ENDPOINT_NAME,
        columns_to_sync=[
            "review_id",
            "gmap_id",
            "business_name",
            "address",
            "rating",
            "time",
            "text"
        ]
    )
    
    review_index.wait_until_ready()
    print(f"✓ Created index: {VECTOR_SEARCH_INDEX_NAME}")

---
### Sync Vector Search Index

In [0]:
# Sync the index after creating or updating the source table
try:
    review_index = vsc.get_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=VECTOR_SEARCH_INDEX_NAME
    )
    print(f"✓ Found existing index: {VECTOR_SEARCH_INDEX_NAME}")
    
    # Check if index is ready
    index_status = review_index.describe()
    if index_status['status']['ready']:
        # Sync only if index is ready
        review_index.sync()
        review_index.wait_until_ready()
        print("✓ Vector Search index is synced and ready.")
    else:
        # Wait for initial sync to complete
        print(f"Index is still initializing (state: {index_status['status']['detailed_state']})...")
        review_index.wait_until_ready()
        print("✓ Vector Search index is ready.")

except Exception:
    print(f"Index not found. Creating index: {VECTOR_SEARCH_INDEX_NAME}")
    
    review_index = vsc.create_delta_sync_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=VECTOR_SEARCH_INDEX_NAME,
        source_table_name=REVIEWS_FOR_VECTOR_TABLE,
        pipeline_type="TRIGGERED",
        primary_key="review_id",
        embedding_source_column="text",
        embedding_model_endpoint_name=EMBEDDING_MODEL_ENDPOINT_NAME,
        columns_to_sync=[
            "review_id",
            "gmap_id",
            "business_name",
            "address",
            "rating",
            "time",
            "text"
        ]
    )
    
    review_index.wait_until_ready()
    print(f"✓ Created and synced index: {VECTOR_SEARCH_INDEX_NAME}")

---
### Helper Function to Parse Vector Search Results

In [0]:
def parse_vector_search_results(results):
    """
    Convert Databricks Vector Search response into a list of dictionaries.
    """
    
    columns = [col["name"] for col in results["manifest"]["columns"]]
    rows = results["result"]["data_array"]
    
    parsed_results = []
    
    for row in rows:
        parsed_results.append(dict(zip(columns, row)))
    
    return parsed_results

---
## Theme Retrieval Tool

In [0]:
%sql
-- Drop existing function if it exists
DROP FUNCTION IF EXISTS main.aai510_final_agent.theme_retrieval;

-- Create SQL UDF for theme retrieval using vector search
CREATE OR REPLACE FUNCTION main.aai510_final_agent.theme_retrieval(
  restaurant_name STRING COMMENT 'Name of the restaurant to retrieve reviews for',
  theme_query STRING COMMENT 'Natural language description of the theme to search for',
  num_results INT COMMENT 'Advisory number (function returns top 20 matches)'
)
RETURNS STRING
COMMENT 'Retrieve reviews related to a user-provided theme using Databricks Vector Search'
RETURN (
  WITH vector_results AS (
    SELECT
      review_id,
      business_name,
      address,
      rating,
      time,
      text
    FROM VECTOR_SEARCH(
      index => 'main.aai510_final_agent.restaurant_review_text_index',
      query => theme_query,
      num_results => 100
    )
    WHERE LOWER(business_name) LIKE CONCAT('%', LOWER(restaurant_name), '%')
    LIMIT 20
  ),
  result_count AS (
    SELECT COUNT(*) as cnt FROM vector_results
  )
  SELECT
    CASE
      WHEN (SELECT cnt FROM result_count) = 0 THEN
        TO_JSON(
          NAMED_STRUCT(
            'status', 'not_found',
            'message', CONCAT('No vector search results found for restaurant \'', restaurant_name, '\' and theme query \'', theme_query, '\''),
            'restaurant_name', restaurant_name,
            'theme_query', theme_query,
            'results', ARRAY()
          )
        )
      ELSE
        TO_JSON(
          NAMED_STRUCT(
            'status', 'success',
            'restaurant_name', restaurant_name,
            'theme_query', theme_query,
            'results', COLLECT_LIST(
              NAMED_STRUCT(
                'review_id', review_id,
                'business_name', business_name,
                'address', address,
                'rating', rating,
                'time', time,
                'text', text
              )
            )
          )
        )
    END
  FROM vector_results
);

In [0]:
%sql
-- Test the theme_retrieval SQL UDF
SELECT main.aai510_final_agent.theme_retrieval(
  'Cesarina',
  'reviews about pasta quality and authentic Italian food',
  5
) as result